# 🚀 Rocket Launch Delay Predictor
## Notebook 1 — Exploratory Data Analysis

This notebook walks through the synthetic launch dataset:
1. Load & inspect
2. Delay rate breakdowns
3. Weather vs outcome analysis
4. Geographic overview
5. Correlation heatmap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
ACCENT = '#58a6ff'
DANGER = '#f85149'
SUCCESS= '#3fb950'

df = pd.read_csv('../data/raw/launches.csv', parse_dates=['launch_date'])
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── Basic stats ───────────────────────────────────────────────────────────────
print('=== Dataset Info ===')
print(f'Total launches:    {len(df):,}')
print(f'Delayed launches:  {df["delayed"].sum():,}  ({df["delayed"].mean():.1%})')
print(f'Date range:        {df["launch_date"].min().date()} → {df["launch_date"].max().date()}')
print(f'Companies:         {df["company"].nunique()}')
print(f'Rockets:           {df["rocket"].nunique()}')
print(f'Sites:             {df["site"].nunique()}')
print()
df.describe()

In [ ]:
# ── Launches over time ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

yearly = df.groupby('launch_year').size()
axes[0].bar(yearly.index, yearly.values, color=ACCENT, alpha=0.8)
axes[0].set_title('Launches Per Year', fontsize=13)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count')

delay_by_year = df.groupby('launch_year')['delayed'].mean()
axes[1].plot(delay_by_year.index, delay_by_year.values * 100, color=DANGER, marker='o', ms=4)
axes[1].set_title('Delay Rate Over Time (%)', fontsize=13)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Delay Rate %')
axes[1].axhline(df['delayed'].mean()*100, linestyle='--', color='gray', alpha=0.6, label='Overall avg')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Delay rate by company & site ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

comp_delay = df.groupby('company')['delayed'].mean().sort_values(ascending=False)
colors_c = [DANGER if v > 0.15 else ('#f0a500' if v > 0.08 else SUCCESS) for v in comp_delay]
axes[0].barh(comp_delay.index, comp_delay.values * 100, color=colors_c, alpha=0.85)
axes[0].set_title('Delay Rate by Company', fontsize=13)
axes[0].set_xlabel('Delay Rate %')
axes[0].axvline(df['delayed'].mean()*100, linestyle='--', color='gray', alpha=0.7)

site_delay = df.groupby('site')['delayed'].mean().sort_values(ascending=False)
axes[1].barh(site_delay.index, site_delay.values * 100, color=ACCENT, alpha=0.85)
axes[1].set_title('Delay Rate by Launch Site', fontsize=13)
axes[1].set_xlabel('Delay Rate %')
axes[1].axvline(df['delayed'].mean()*100, linestyle='--', color='gray', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# ── Weather distributions ─────────────────────────────────────────────────────
weather_cols = ['wind_speed_kmh', 'temp_celsius', 'precipitation_mm', 'cloud_cover_pct']
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.flatten()

for ax, col in zip(axes, weather_cols):
    on_time = df[df['delayed']==0][col]
    delayed = df[df['delayed']==1][col]
    ax.hist(on_time, bins=40, alpha=0.6, color=SUCCESS, label='On Time', density=True)
    ax.hist(delayed, bins=40, alpha=0.6, color=DANGER,  label='Delayed', density=True)
    ax.set_title(col.replace('_',' ').title(), fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Weather Feature Distributions: Delayed vs On Time', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
num_cols = ['rocket_age_years','provider_success_rate','site_success_rate',
            'launches_this_month','temp_celsius','wind_speed_kmh',
            'precipitation_mm','cloud_cover_pct','is_crewed','delayed']

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, ax=ax, linewidths=0.4,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Delay hours distribution ──────────────────────────────────────────────────
delayed_df = df[df['delayed']==1]
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(delayed_df['delay_hours'], bins=60, color='#f0a500', alpha=0.85, edgecolor='none')
ax.axvline(delayed_df['delay_hours'].median(), color=DANGER, linewidth=1.5,
           linestyle='--', label=f"Median: {delayed_df['delay_hours'].median():.1f}h")
ax.set_title('Distribution of Delay Duration (hours)', fontsize=13)
ax.set_xlabel('Delay (hours)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

print(delayed_df['delay_hours'].describe())

In [ ]:
# ── Key takeaways ─────────────────────────────────────────────────────────────
print('=== EDA Summary Takeaways ===')
print(f'1. Overall delay rate: {df["delayed"].mean():.1%}')
print(f'2. Highest-risk company: {comp_delay.index[0]} ({comp_delay.iloc[0]:.1%})')
print(f'3. Lowest-risk company:  {comp_delay.index[-1]} ({comp_delay.iloc[-1]:.1%})')
print(f'4. Avg wind speed (delayed): {df[df.delayed==1].wind_speed_kmh.mean():.1f} km/h')
print(f'   Avg wind speed (on-time): {df[df.delayed==0].wind_speed_kmh.mean():.1f} km/h')
print(f'5. Median delay duration:    {delayed_df["delay_hours"].median():.1f} hours')
print(f'6. Max delay in dataset:     {delayed_df["delay_hours"].max():.0f} hours')